# Inspect rows with `domains_visited > 0`

This notebook reads a CSV file with verification results and shows **exactly which rows entered Branch-and-Bound**, i.e. where `domains_visited > 0`.

**What you get:**
- Filtered table of rows where `domains_visited > 0`
- Clean view of key columns
- Optional export to a new CSV
- Quick summary counts


In [1]:
import pandas as pd

## Load CSV
Update the path to point to your CSV file.

In [14]:
CSV_PATH = "../results/all_2eps_2pert/results.csv"

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()

Shape: (5617, 10)


,instance_id,onnx,vnnlib,timeout,result,lb_minus_rhs,domains_visited,bab_time,all_time,init_unstable
0,1,onnx/vgg16-7.onnx,vnnlib/n01440764_tench_global_k10_eps_0.0001.v...,1200,sat False,-inf,NaN,1.485778,2251.853036,0.0
1,2,onnx/vgg16-7.onnx,vnnlib/n01440764_tench_global_k50176_eps_0.000...,1200,sat True,NaN,NaN,NaN,2242.709431,0.0
2,3,onnx/vgg16-7.onnx,vnnlib/n01440764_tench_global_k10_eps_0.0003.v...,1200,sat False,-inf,NaN,2.445933,1859.241091,0.0
3,4,onnx/vgg16-7.onnx,vnnlib/n01440764_tench_global_k50176_eps_0.000...,1200,sat True,NaN,NaN,NaN,1860.112109,0.0
4,5,onnx/vgg16-7.onnx,vnnlib/n01440764_tench_seg0_fixmask_k10_eps_0....,1200,sat False,-inf,NaN,1.590552,2182.993772,0.0


## Sanity checks
Make sure `domains_visited` exists and is numeric.

In [15]:
assert "domains_visited" in df.columns, "domains_visited column not found!"

df["domains_visited"] = pd.to_numeric(df["domains_visited"], errors="coerce").fillna(0)

## Filter rows that entered BnB
These are the rows where **Branch-and-Bound was actually used**.

In [18]:
bnb_df = df[df["domains_visited"] > 0].copy()

bnb_df = bnb_df.sort_values(
    by="lb_minus_rhs",
    ascending=False
)

print("Rows with domains_visited > 0:", len(bnb_df))
bnb_df.head()

Rows with domains_visited > 0: 290


,instance_id,onnx,vnnlib,timeout,result,lb_minus_rhs,domains_visited,bab_time,all_time,init_unstable
3934,4406,onnx/vgg16-7.onnx,vnnlib/n02782093_balloon_global_k50176_eps_0.0...,1200,timeout False,-0.000140,3.0,2105.254942,2119.689082,0.0
1465,1710,onnx/vgg16-7.onnx,vnnlib/n02089973_English_foxhound_seg0_fixmask...,1200,timeout False,-0.005404,3.0,1995.865872,2014.198090,0.0
4276,4784,onnx/vgg16-7.onnx,vnnlib/n02869837_bonnet_seg0_fixmask_k50176_ep...,1200,timeout False,-0.006610,3.0,1890.326157,1905.408218,0.0
3215,3618,onnx/vgg16-7.onnx,vnnlib/n02408429_water_buffalo_seg0_fixmask_k5...,1200,timeout False,-0.008544,3.0,2230.481907,2244.842886,0.0
1572,1834,onnx/vgg16-7.onnx,vnnlib/n02093256_Staffordshire_bullterrier_seg...,1200,timeout False,-0.010622,3.0,2167.317146,2182.244877,0.0


## Show key columns (clean view)

In [19]:
cols = [
    "instance_id",
    "image",
    "tag",
    "is_global",
    "segment_index",
    "eps",
    "k",
    "result",
    "lb_minus_rhs",
    "domains_visited",
    "bab_time",
    "all_time",
]

# Only keep columns that exist (safe if your CSV is slightly different)
cols = [c for c in cols if c in bnb_df.columns]

bnb_df[cols].sort_values("domains_visited", ascending=False)

,instance_id,result,lb_minus_rhs,domains_visited,bab_time,all_time
3934,4406,timeout False,-1.401901e-04,3.0,2105.254942,2119.689082
2256,2566,timeout False,-3.183642e+02,3.0,2674.816055,2692.061052
2933,3314,timeout False,-1.272306e+02,3.0,2627.125324,2641.579709
5311,6782,timeout False,-1.228938e+02,3.0,4710.721872,4764.511176
4335,4846,timeout False,-1.187914e+02,3.0,2826.484781,2840.766172
...,...,...,...,...,...,...
5474,8782,timeout False,-6.321603e+00,3.0,4314.908828,4368.216840
1272,1496,timeout False,-6.010989e+00,3.0,2403.408789,2418.558656
1658,1928,timeout False,-5.858961e+00,3.0,2298.208335,2312.723564
5299,6770,timeout False,-5.714836e+00,3.0,3621.238371,3636.907764


## Optional: save to a new CSV

In [20]:
OUT_PATH = "rows_with_bnb_entered.csv"
bnb_df.to_csv(OUT_PATH, index=False)
print("Saved to", OUT_PATH)

Saved to rows_with_bnb_entered.csv


## Optional: quick summary
How many BnB vs non-BnB rows?

In [ ]:
summary = (
    df.assign(bnb_entered=df["domains_visited"] > 0)
      .groupby("bnb_entered")
      .size()
      .rename("count")
)

summary